# Olist ETL / Transform (revised)

Changes compared with the previous version:

1. `max_payment_installments` now uses `.max()` (was `.sum()`).
2. `total_product_weight_g` uses `sum(min_count=1)` so orders with all-missing weights stay missing (was silently 0).
3. `is_delayed` is compared at **date level** (the estimated delivery date has no time part, so same-day deliveries were wrongly labelled as delayed).
4. Geolocation records outside Brazil's extent are removed before averaging coordinates per zip code prefix.
5. `n_unique_payment_types` now counts `payment_type` (it counted `payment_installments`).
6. `item_value_features` is no longer concatenated twice.
7. No variable is re-used under a different type (the old `product_features` list overwrote a DataFrame).

Run with **Restart Kernel and Run All**.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_PATH = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/Raw")
PROCESSED_PATH = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# New file name so the old parquet stays available for before/after comparison.
OUTPUT_FILE = PROCESSED_PATH / "final_df_(distance)_v2.parquet"

# Approximate bounding box of Brazil (used to drop wrong geolocation records)
BRAZIL_LAT = (-33.75, 5.27)
BRAZIL_LNG = (-73.99, -34.79)
EARTH_RADIUS_KM = 6371

## 1. Import files

Only the seven tables that feed the features are loaded (reviews and the category translation table are not used).

In [2]:
orders = pd.read_csv(RAW_PATH / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW_PATH / "olist_customers_dataset.csv")
order_items = pd.read_csv(RAW_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_PATH / "olist_sellers_dataset.csv")
payments = pd.read_csv(RAW_PATH / "olist_order_payments_dataset.csv")
geolocation = pd.read_csv(RAW_PATH / "olist_geolocation_dataset.csv")

for name, frame in {
    "orders": orders, "customers": customers, "order_items": order_items,
    "products": products, "sellers": sellers, "payments": payments,
    "geolocation": geolocation,
}.items():
    print(f"{name}: {frame.shape}")

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
products: (32951, 9)
sellers: (3095, 4)
payments: (103886, 5)
geolocation: (1000163, 5)


## 2. Data type conversion

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## 3. Data quality cleaning

In [4]:
print("order_id unique:", orders["order_id"].is_unique)
print("delivered < purchase:", (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print("approved < purchase:", (orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum())

invalid_delivery = (
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
)
print("delivered_customer < delivered_carrier:", invalid_delivery.sum())

order_id unique: True
delivered < purchase: 0
approved < purchase: 0
delivered_customer < delivered_carrier: 23


Data quality rule: if `order_delivered_customer_date < order_delivered_carrier_date`, the order is temporally inconsistent and excluded.

In [5]:
n_before = len(orders)
orders = orders.loc[~invalid_delivery].copy()
print(f"orders: {n_before} -> {len(orders)} (removed {n_before - len(orders)})")

print("negative price:", (order_items["price"] < 0).sum())
print("negative freight:", (order_items["freight_value"] < 0).sum())

orders: 99441 -> 99418 (removed 23)
negative price: 0
negative freight: 0


## 4. Order-level features

### 4.1 Items

In [6]:
item_features = order_items.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight_value=("freight_value", "sum"),
    n_unique_products=("product_id", "nunique"),
    n_unique_sellers=("seller_id", "nunique"),
)
item_features.head()

,item_count,total_price,total_freight_value,n_unique_products,n_unique_sellers
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1
00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1
000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1
00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1


### 4.2 Products

`sum(min_count=1)` keeps the weight missing when every product of the order has a missing weight.

In [7]:
order_item_products = order_items[["order_id", "product_id"]].merge(
    products, on="product_id", how="left"
)
grouped_products = order_item_products.groupby("order_id")

product_features_agg = pd.concat(
    [
        grouped_products["product_weight_g"].sum(min_count=1).rename("total_product_weight_g"),
        grouped_products["product_length_cm"].mean().rename("average_product_length_cm"),
        grouped_products["product_height_cm"].mean().rename("average_product_height_cm"),
        grouped_products["product_width_cm"].mean().rename("average_product_width_cm"),
        grouped_products["product_category_name"].nunique().rename("n_unique_categories"),
    ],
    axis=1,
)
product_features_agg.head()

,total_product_weight_g,average_product_length_cm,average_product_height_cm,average_product_width_cm,n_unique_categories
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,650.0,28.0,9.0,14.0,1
00018f77f2f0320c557190d7a144bdd3,30000.0,50.0,30.0,40.0,1
000229ec398224ef6ca0657da4fc703e,3050.0,33.0,13.0,33.0,1
00024acbcdf0a6daa1e931b038114c75,200.0,16.0,10.0,15.0,1
00042b26cf59d7ce69dfabb4e55b4fd9,3750.0,35.0,40.0,30.0,1


### 4.3 Sellers

In [8]:
order_item_sellers = order_items[["order_id", "seller_id"]].merge(
    sellers, on="seller_id", how="left"
)
grouped_sellers = order_item_sellers.groupby("order_id")

seller_features_agg = pd.concat(
    [
        grouped_sellers["seller_city"].nunique().rename("n_unique_seller_city"),
        grouped_sellers["seller_state"].nunique().rename("n_unique_seller_state"),
    ],
    axis=1,
)
seller_features_agg.head()

,n_unique_seller_city,n_unique_seller_state
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,1,1
00018f77f2f0320c557190d7a144bdd3,1,1
000229ec398224ef6ca0657da4fc703e,1,1
00024acbcdf0a6daa1e931b038114c75,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,1


### 4.4 Payments

`max_payment_installments` is the longest instalment plan among the payment records of an order.

In [9]:
grouped_payments = payments.groupby("order_id")

payment_features_agg = pd.concat(
    [
        grouped_payments["payment_value"].sum().rename("total_payment_value"),
        grouped_payments["payment_installments"].max().rename("max_payment_installments"),
        grouped_payments["payment_type"].nunique().rename("n_unique_payment_types"),
    ],
    axis=1,
)
payment_features_agg.head()

,total_payment_value,max_payment_installments,n_unique_payment_types
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
00018f77f2f0320c557190d7a144bdd3,259.83,3,1
000229ec398224ef6ca0657da4fc703e,216.87,5,1
00024acbcdf0a6daa1e931b038114c75,25.78,2,1
00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


### 4.5 Customers

In [10]:
customer_features = customers[["customer_id", "customer_city", "customer_state"]]
print("customer_id unique:", customer_features["customer_id"].is_unique)

customer_id unique: True


## 5. Distance feature

Steps: (1) keep geolocation records inside Brazil, (2) average the coordinates per zip code prefix, (3) join coordinates to seller and customer zip code prefixes, (4) Haversine distance for every order item, (5) average within each order.

In [11]:
inside_brazil = (
    geolocation["geolocation_lat"].between(*BRAZIL_LAT)
    & geolocation["geolocation_lng"].between(*BRAZIL_LNG)
)
print(f"geolocation records outside Brazil removed: {(~inside_brazil).sum()} of {len(geolocation)}")

zip_geo = (
    geolocation.loc[inside_brazil]
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
)
print("zip code prefixes with a coordinate:", len(zip_geo))

geolocation records outside Brazil removed: 42 of 1000163
zip code prefixes with a coordinate: 19010


In [12]:
seller_points = (
    order_item_sellers[["order_id", "seller_id", "seller_zip_code_prefix"]]
    .merge(
        zip_geo.rename(columns={"lat": "seller_lat", "lng": "seller_lng"}),
        left_on="seller_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left",
    )
    .drop(columns="geolocation_zip_code_prefix")
)

customer_points = (
    orders[["order_id", "customer_id"]]
    .merge(customers[["customer_id", "customer_zip_code_prefix"]], on="customer_id", how="left")
    .merge(
        zip_geo.rename(columns={"lat": "customer_lat", "lng": "customer_lng"}),
        left_on="customer_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left",
    )
    .drop(columns="geolocation_zip_code_prefix")
)

print("order_id unique in customer_points:", customer_points["order_id"].is_unique)

order_distance = seller_points.merge(
    customer_points[["order_id", "customer_lat", "customer_lng"]],
    on="order_id",
    how="left",
    validate="many_to_one",
)
print("order-item rows:", len(order_distance), "| order_items rows:", len(order_items))
order_distance[["seller_lat", "seller_lng", "customer_lat", "customer_lng"]].isna().sum()

order_id unique in customer_points: True
order-item rows: 112650 | order_items rows: 112650


seller_lat      253
seller_lng      253
customer_lat    353
customer_lng    353
dtype: int64

In [13]:
valid_distance = order_distance.dropna(
    subset=["seller_lat", "seller_lng", "customer_lat", "customer_lng"]
).copy()


def haversine_km(lat1, lng1, lat2, lng2, radius=EARTH_RADIUS_KM):
    # d = 2R * arcsin( sqrt( sin^2(dphi/2) + cos(phi1) cos(phi2) sin^2(dlambda/2) ) )
    phi1, lam1 = np.radians(lat1), np.radians(lng1)
    phi2, lam2 = np.radians(lat2), np.radians(lng2)
    a = (
        np.sin((phi2 - phi1) / 2) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin((lam2 - lam1) / 2) ** 2
    )
    return 2 * radius * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


valid_distance["distance_km"] = haversine_km(
    valid_distance["customer_lat"], valid_distance["customer_lng"],
    valid_distance["seller_lat"], valid_distance["seller_lng"],
)
print(valid_distance["distance_km"].describe())
print("distance > 4500 km:", (valid_distance["distance_km"] > 4500).sum())

average_distance_km = (
    valid_distance.groupby("order_id")["distance_km"].mean().rename("average_distance_km")
)
average_distance_km.describe()

count    112045.000000
mean        596.707249
std         588.801608
min           0.000000
25%         184.235602
50%         431.607976
75%         792.236663
max        3579.894251
Name: distance_km, dtype: float64
distance > 4500 km: 0


count    98153.000000
mean       601.352388
std        593.570166
min          0.000000
25%        187.344597
50%        433.779287
75%        798.286950
max       3579.894251
Name: average_distance_km, dtype: float64

## 6. Assemble the order-level dataset

In [14]:
final_df = orders.copy()

for name, features, key in [
    ("item_features", item_features, "order_id"),
    ("product_features_agg", product_features_agg, "order_id"),
    ("seller_features_agg", seller_features_agg, "order_id"),
    ("payment_features_agg", payment_features_agg, "order_id"),
    ("average_distance_km", average_distance_km, "order_id"),
]:
    final_df = final_df.merge(features, on=key, how="left", validate="one_to_one")
    print(f"after {name}: {final_df.shape}")

final_df = final_df.merge(customer_features, on="customer_id", how="left", validate="many_to_one")
print("after customer_features:", final_df.shape)
print("order_id unique:", final_df["order_id"].is_unique)

after item_features: (99418, 13)
after product_features_agg: (99418, 18)
after seller_features_agg: (99418, 20)
after payment_features_agg: (99418, 23)
after average_distance_km: (99418, 24)
after customer_features: (99418, 26)
order_id unique: True


## 7. Target

`is_delayed` is compared at date level: `order_estimated_delivery_date` only has a date (00:00:00), so a delivery on the estimated day is on time. Undelivered orders stay missing.

In [15]:
final_df["delivery_delay_days"] = (
    final_df["order_delivered_customer_date"] - final_df["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

final_df["is_delayed"] = (
    final_df["order_delivered_customer_date"].dt.normalize()
    > final_df["order_estimated_delivery_date"].dt.normalize()
).astype("boolean")

final_df.loc[final_df["order_delivered_customer_date"].isna(), "is_delayed"] = pd.NA

print(final_df["is_delayed"].value_counts(dropna=False))
print(final_df["is_delayed"].value_counts(normalize=True, dropna=True))

is_delayed
False    89918
True      6535
<NA>      2965
Name: count, dtype: Int64
is_delayed
False    0.932247
True     0.067753
Name: proportion, dtype: Float64


## 8. Checks and numbers for the report

In [16]:
final_features = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km",
]
target = "is_delayed"

missing = final_df[final_features + [target]].isna().sum()
print("Missing values before dropna (M = %d orders):" % len(final_df))
print(missing)

model_df = final_df.dropna(subset=final_features + [target]).copy()
n_all, n_model = len(final_df), len(model_df)
print()
print(f"M (before removal): {n_all}")
print(f"N (complete cases): {n_model}")
print(f"removed: {n_all - n_model} ({(n_all - n_model) / n_all:.1%})")
print("max distance (km):", round(final_df["average_distance_km"].max(), 1))
print("is_delayed share in model_df:")
print(model_df[target].value_counts(normalize=True))

Missing values before dropna (M = 99418 orders):
total_price                   775
total_freight_value           775
total_product_weight_g        791
average_product_length_cm     791
average_product_height_cm     791
item_count                    775
max_payment_installments        1
average_distance_km          1265
is_delayed                   2965
dtype: int64

M (before removal): 99418
N (complete cases): 95959
removed: 3459 (3.5%)
max distance (km): 3579.9
is_delayed share in model_df:
is_delayed
False    0.932377
True     0.067623
Name: proportion, dtype: Float64


In [17]:
# Which order statuses have a missing is_delayed? (to describe why they are removed)
final_df.loc[final_df["is_delayed"].isna(), "order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

## 9. Save

In [18]:
final_df.to_parquet(OUTPUT_FILE, index=False)

loaded_df = pd.read_parquet(OUTPUT_FILE)
print(loaded_df.shape)
print(loaded_df.columns.tolist())

(99418, 28)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_price', 'total_freight_value', 'n_unique_products', 'n_unique_sellers', 'total_product_weight_g', 'average_product_length_cm', 'average_product_height_cm', 'average_product_width_cm', 'n_unique_categories', 'n_unique_seller_city', 'n_unique_seller_state', 'total_payment_value', 'max_payment_installments', 'n_unique_payment_types', 'average_distance_km', 'customer_city', 'customer_state', 'delivery_delay_days', 'is_delayed']


In [19]:
# 1. 订单表没有重复，且商品行没有被合并放大
print(final_df["order_id"].is_unique, len(final_df))

# 2. 重量、尺寸、价格里有没有 0 或异常值
print((products["product_weight_g"] == 0).sum())
print(final_df[["total_price", "total_freight_value", "total_product_weight_g",
                "average_product_length_cm", "average_product_height_cm"]].describe().T)

# 3. 分期数有没有 0
print((payments["payment_installments"] == 0).sum())

# 4. 状态是 delivered 但没有送达日期的订单（应该是 8 个）
print(((final_df["order_status"] == "delivered")
       & final_df["order_delivered_customer_date"].isna()).sum())


True 99418
4
                             count         mean          std   min     25%  \
total_price                98643.0   137.748091   210.665857  0.85   45.90   
total_freight_value        98643.0    22.820997    21.650033  0.00   13.85   
total_product_weight_g     98627.0  2389.823091  4772.610086  0.00  300.00   
average_product_length_cm  98627.0    30.100121    16.080625  7.00   18.00   
average_product_height_cm  98627.0    16.484491    13.282737  2.00    8.00   

                              50%      75%        max  
total_price                 86.90   149.90   13440.00  
total_freight_value         17.17    24.03    1794.96  
total_product_weight_g     750.00  2065.00  184400.00  
average_product_length_cm   25.00    38.00     105.00  
average_product_height_cm   13.00    20.00     105.00  
2
8
